<a href="https://colab.research.google.com/github/jaalvalcan/GEE_index_sets/blob/main/Deepforest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install necessary libraries
!pip install deepforest rasterio pyproj leafmap localtileserver -q

import pandas as pd
import rasterio
from pyproj import Transformer
from deepforest import main
import leafmap.leafmap as leafmap
from google.colab import drive
import numpy as np
from PIL import Image
import os

# --- THE FIX FOR DecompressionBombError ---
Image.MAX_IMAGE_PIXELS = None

# 2. Mount Drive
drive.mount('/content/drive')
image_input = '/content/drive/MyDrive/bra.tif'

# 3. Initialize DeepForest
print("🌲 Initializing DeepForest Model...")
model = main.deepforest()

# 4. Run Prediction in Tiled Mode
print("🚁 Processing high-res imagery (Tiling mode)...")
try:
    predictions = model.predict_tile(path=image_input, patch_size=800, patch_overlap=0.1)
except TypeError:
    print("🔄 Retrying with positional argument...")
    predictions = model.predict_tile(image_input, patch_size=800, patch_overlap=0.1)

# 5. Transform to EPSG:4326 (WGS 84)
if predictions is not None and not predictions.empty:
    print("🌍 Converting detections to WGS 84 (Degrees)...")
    with rasterio.open(image_input) as src:
        img_crs = src.crs
        img_transform = src.transform

        predictions['center_x_pixel'] = (predictions['xmin'] + predictions['xmax']) / 2
        predictions['center_y_pixel'] = (predictions['ymin'] + predictions['ymax']) / 2

        xs, ys = rasterio.transform.xy(img_transform, predictions['center_y_pixel'], predictions['center_x_pixel'])

        transformer = Transformer.from_crs(img_crs, "epsg:4326", always_xy=True)
        predictions['wgs_lon'], predictions['wgs_lat'] = transformer.transform(np.array(xs), np.array(ys))

    # --- FIX: Bin the score into 5 levels to avoid the ValueError ---
    # This groups the scores into discrete levels like 'Low', 'Medium', 'High'
    predictions['score_level'] = pd.cut(
        predictions['score'],
        bins=5,
        labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
    )
    # ----------------------------------------------------------------

    print(f"✅ Success! Found {len(predictions)} tree crowns.")

    # 6. Save Results to Drive
    save_path_csv = '/content/drive/MyDrive/deepforest_trees_wgs84.csv'
    predictions[['wgs_lat', 'wgs_lon', 'score', 'score_level']].to_csv(save_path_csv, index=False)
    print(f"💾 File saved to: {save_path_csv}")

    # 7. Visualize on the Map
    m = leafmap.Map(center=[predictions['wgs_lat'].mean(), predictions['wgs_lon'].mean()], zoom=19)
    m.add_raster(image_input, layer_name="Drone Imagery")

    # Use the NEW 'score_level' column for the color_column
    m.add_points_from_xy(
        predictions,
        x="wgs_lon",
        y="wgs_lat",
        layer_name="DeepForest Tree Crowns",
        color_column='score_level', # <--- FIXED
        colormap='YlGn',
        radius=5
    )
    display(m)

else:
    print("❌ No trees were detected.")